In [2]:
import asyncio
import json

import websockets
from pynq import MMIO

# -----------------------------------
# AXI GPIO / MMIO
# -----------------------------------

BASE_ADDR = 0x41200000
ADDR_RANGE = 0x1000

mmio = MMIO(BASE_ADDR, ADDR_RANGE)

# -----------------------------------
# WebSocket clients
# -----------------------------------

clients = set()

# -----------------------------------
# FSM decoding
# -----------------------------------

STATE_MAP = {
    0: "IDLE",
    1: "FILL",
    2: "WASH",
    3: "RINSE",
    4: "SPIN",
    5: "DONE",
}

# -----------------------------------
# WebSocket handler
# -----------------------------------

async def ws_handler(websocket):
    clients.add(websocket)

    print("Client connected")

    try:
        await websocket.wait_closed()

    finally:
        clients.remove(websocket)
        print("Client disconnected")

# -----------------------------------
# Broadcast helper
# -----------------------------------

async def broadcast(msg: str):
    dead = []

    for ws in clients:
        try:
            await ws.send(msg)
        except:
            dead.append(ws)

    for ws in dead:
        clients.remove(ws)

# -----------------------------------
# FSM polling loop
# -----------------------------------

async def fsm_loop():

    previous_state = None

    while True:

        raw_value = mmio.read(0)

        state = STATE_MAP.get(raw_value, "UNKNOWN")

        if state != previous_state:

            print(f"FSM -> {state}")

            payload = {
                "type": "state",
                "value": state
            }

            await broadcast(json.dumps(payload))

            previous_state = state

        await asyncio.sleep(0.05)

# -----------------------------------
# Main
# -----------------------------------

async def main():

    server = await websockets.serve(
        ws_handler,
        "0.0.0.0",
        8765
    )

    print("WebSocket server running")

    asyncio.create_task(fsm_loop())

    await server.wait_closed()

await main()

WebSocket server running
FSM -> IDLE
FSM -> FILL
FSM -> WASH
FSM -> RINSE
FSM -> SPIN
FSM -> DONE
FSM -> IDLE


CancelledError: 